# Phase 1.1 — Dataset Acquisition Verification

This notebook verifies the raw Retailrocket dataset without loading the full ~1 GB dataset into memory.

**Expected location:** `../data/raw/` relative to this notebook.

This is an audit/verification notebook only. No raw files are modified.

In [1]:
from pathlib import Path
import csv
import hashlib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

print('Project root :', PROJECT_ROOT.resolve())
print('Raw data dir:', RAW_DIR.resolve())
print('Exists      :', RAW_DIR.exists())

Project root : F:\annuspeaks.com\recommendation-system
Raw data dir: F:\annuspeaks.com\recommendation-system\data\raw
Exists      : True


In [2]:
expected_files = [
    'category_tree.csv',
    'events.csv',
    'item_properties_part1.csv',
    'item_properties_part2.csv',
]

print('RAW DATA INVENTORY')
print('-' * 80)

for name in expected_files:
    path = RAW_DIR / name
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f'✓ {name:<32} {size_mb:>10.2f} MB')
    else:
        print(f'✗ MISSING: {name}')

RAW DATA INVENTORY
--------------------------------------------------------------------------------
✓ category_tree.csv                      0.01 MB
✓ events.csv                            89.87 MB
✓ item_properties_part1.csv            461.88 MB
✓ item_properties_part2.csv            389.99 MB


In [3]:
def csv_header(path):
    with path.open('r', encoding='utf-8', errors='replace', newline='') as f:
        return next(csv.reader(f))

print('CSV SCHEMAS / HEADERS')
print('-' * 80)

for name in expected_files:
    path = RAW_DIR / name
    if path.exists():
        print(f'\n{name}')
        print(csv_header(path))

CSV SCHEMAS / HEADERS
--------------------------------------------------------------------------------

category_tree.csv
['categoryid', 'parentid']

events.csv
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

item_properties_part1.csv
['timestamp', 'itemid', 'property', 'value']

item_properties_part2.csv
['timestamp', 'itemid', 'property', 'value']


In [4]:
def count_rows(path, chunk_size=1024 * 1024):
    """Count physical CSV data rows without loading the file into memory."""
    total_newlines = 0
    with path.open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            total_newlines += chunk.count(b'\n')
    return max(total_newlines - 1, 0)

print('ROW COUNTS')
print('-' * 80)

row_counts = {}
for name in expected_files:
    path = RAW_DIR / name
    if path.exists():
        rows = count_rows(path)
        row_counts[name] = rows
        print(f'{name:<32} {rows:,} rows')

ROW COUNTS
--------------------------------------------------------------------------------
category_tree.csv                1,669 rows
events.csv                       2,756,101 rows
item_properties_part1.csv        10,999,999 rows
item_properties_part2.csv        9,275,903 rows


In [5]:
print('SAMPLE RECORDS')
print('-' * 80)

for name in expected_files:
    path = RAW_DIR / name
    if path.exists():
        print(f'\n### {name}')
        try:
            sample = pd.read_csv(path, nrows=5)
            display(sample)
        except Exception as exc:
            print('Could not parse sample:', repr(exc))

SAMPLE RECORDS
--------------------------------------------------------------------------------

### category_tree.csv


,categoryid,parentid
0,1016,213
1,809,169
2,570,9
3,1691,885
4,536,1691



### events.csv


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN



### item_properties_part1.csv


,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513



### item_properties_part2.csv


,timestamp,itemid,property,value
0,1433041200000,183478,561,769062
1,1439694000000,132256,976,n26.400 1135780
2,1435460400000,420307,921,1149317 1257525
3,1431831600000,403324,917,1204143
4,1435460400000,230701,521,769062


In [6]:
print('EVENT TYPES / TIMESTAMP RANGE')
print('-' * 80)

events_path = RAW_DIR / 'events.csv'
if events_path.exists():
    events_sample = pd.read_csv(events_path, nrows=200_000)
    print('Columns:', list(events_sample.columns))
    if 'event' in events_sample.columns:
        print('\nEvent distribution in audit sample:')
        print(events_sample['event'].value_counts(dropna=False))
    if 'timestamp' in events_sample.columns:
        ts = pd.to_numeric(events_sample['timestamp'], errors='coerce')
        print('\nTimestamp range in audit sample:')
        print('min:', ts.min())
        print('max:', ts.max())

EVENT TYPES / TIMESTAMP RANGE
--------------------------------------------------------------------------------
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

Event distribution in audit sample:
event
view           193762
addtocart        4675
transaction      1563
Name: count, dtype: int64

Timestamp range in audit sample:
min: 1433138409198
max: 1434002363930


In [7]:
print('BASIC INTEGRITY CHECKS')
print('-' * 80)

for name in expected_files:
    path = RAW_DIR / name
    if path.exists():
        size = path.stat().st_size
        print(f'✓ {name}: non-empty ({size:,} bytes)') if size > 0 else print(f'✗ {name}: EMPTY')

unexpected = sorted(p.name for p in RAW_DIR.iterdir() if p.is_file() and p.suffix.lower() == '.csv' and p.name not in expected_files)
print('\nUnexpected CSV files:', unexpected if unexpected else 'None')

BASIC INTEGRITY CHECKS
--------------------------------------------------------------------------------
✓ category_tree.csv: non-empty (14,454 bytes)
✓ events.csv: non-empty (94,237,913 bytes)
✓ item_properties_part1.csv: non-empty (484,315,749 bytes)
✓ item_properties_part2.csv: non-empty (408,929,907 bytes)

Unexpected CSV files: None


## Verification Result

After running all cells, confirm:

- All four expected raw CSV files exist.
- File sizes are non-zero and plausible.
- Headers parse successfully.
- Row counts are obtainable.
- Sample records parse successfully.
- `events.csv` contains the expected behavioral event field/signals.
- No raw files were modified by this notebook.

**Do not move anything into `data/processed/` yet.** That directory is reserved for derived data after the inventory and data-engineering decisions are made.